In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# 2.3 Recolección y Preparación de Datos

Primero creamos el dataframe DfStudPerf con la información de StudentsPerformance.csv que hemos descargado de Kaggle. 

In [33]:
DfStudPerf=pd.read_csv('C:/Users/Norberto/OneDrive/Documentos/UCAECE/Seminario de Datos 1/StudentsPerformance.csv')

## Revisión inicial del dataset

> fuente o link del dataset

Link: https://www.kaggle.com/datasets/spscientist/students-performance-in-exams



In [34]:
DfStudPerf.info() #este comando nos brinda la información principal del df, la cantidad de filas, las columnas con su tipo y cantidad de valores nulos.

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   gender                       1000 non-null   str  
 1   race/ethnicity               1000 non-null   str  
 2   parental level of education  1000 non-null   str  
 3   lunch                        1000 non-null   str  
 4   test preparation course      1000 non-null   str  
 5   math score                   1000 non-null   int64
 6   reading score                1000 non-null   int64
 7   writing score                1000 non-null   int64
dtypes: int64(3), str(5)
memory usage: 62.6 KB


In [35]:
#Aquí encontramos los valores únicos que nos permitirán saber los subgrupos de cada columna.

Generos= DfStudPerf['gender'].unique()
print(Generos)

GruposEtnicos= DfStudPerf['race/ethnicity'].unique()
print(GruposEtnicos)

NivelEducPadres= DfStudPerf['parental level of education'].unique()
print(NivelEducPadres)

TiposdeAlmuerzo= DfStudPerf['lunch'].unique()
print(TiposdeAlmuerzo)

CursodeIngreso = DfStudPerf['test preparation course'].unique()
print(CursodeIngreso)

<StringArray>
['female', 'male']
Length: 2, dtype: str
<StringArray>
['group B', 'group C', 'group A', 'group D', 'group E']
Length: 5, dtype: str
<StringArray>
[ 'bachelor's degree',       'some college',    'master's degree',
 'associate's degree',        'high school',   'some high school']
Length: 6, dtype: str
<StringArray>
['standard', 'free/reduced']
Length: 2, dtype: str
<StringArray>
['none', 'completed']
Length: 2, dtype: str


> Dimenciones del dataset

* filas: 1000
* Columnas: 8

> Variables

* gender: género del estudiante (femenino o masculino)
* race/ethnicity: grupo étnico del estudiante que por ser un dato sensible preserva su anonimato. Hay 5 grupos (A,B,C,D y E) .
* parental level of education: nivel educativo de los padres del estudiante. Incluye categorías como high school, some college, associate’s degree, bachelor’s degree y master’s degree.
* lunch: tipo de alimentación del estudiante. Puede ser standard o free/reduced(subsidiada).
* test preparation course: indica si el estudiante completó un curso de preparación (completed) o no lo realizó (none).
* math score: puntaje obtenido en el examen de matemática, en una escala de 0 a 100.
* reading score: puntaje obtenido en el examen de lectura, en una escala de 0 a 100.
* writing score: puntaje obtenido en el examen de escritura, en una escala de 0 a 100.

> Tipos de Datos

* gender: string
* race/ethnicity: string 
* parental level of education: string 
* lunch: string
* test preparation course: string
* math score: int64
* reading score: int64
* writing score: int64

> Nueva variable derivada: Promedio 

Se crea la variable *avg_scr*, promedio resultante entre las tres notas de cada alumno (math score, reading score y writing score). Representa el rendimiento académico general del estudiante.

In [36]:
DfStudPerf['avg_scr'] = (
    DfStudPerf['math score'] +
    DfStudPerf['reading score'] +
    DfStudPerf['writing score']
) / 3

print(DfStudPerf['avg_scr'].head(10))

0    72.666667
1    82.333333
2    92.666667
3    49.333333
4    76.333333
5    77.333333
6    91.666667
7    40.666667
8    65.000000
9    49.333333
Name: avg_scr, dtype: float64


> Estadísticas Descriptivas

DfStudPerf.describe() muestra los estadísticos de las columnas con valores continuos (math score, reading score y writing score, y la variable derivada avg_scr) del dataframe.

In [37]:
DfStudPerf.describe()

,math score,reading score,writing score,avg_scr
count,1000.00000,1000.000000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000,67.770667
std,15.16308,14.600192,15.195657,14.257326
min,0.00000,17.000000,10.000000,9.000000
25%,57.00000,59.000000,57.750000,58.333333
50%,66.00000,70.000000,69.000000,68.333333
75%,77.00000,79.000000,79.000000,77.666667
max,100.00000,100.000000,100.000000,100.000000


# 2.4. Modelado Básico

## Selección de variables

> Variable Objetivo (y)
* avg_scr

> Variable predictoria (X)
* test preparation course
* parental level of education
* lunch
* gender
* race/ethnicity

Las variables predictorias (X) son categóricas, por eso se utilizará *get_dummies()* para convertirlas en variables binarias O/1. 

In [58]:
# Definimos las variables:
X = DfStudPerf[['test preparation course',
        'parental level of education',
        'lunch',
        'gender',
        'race/ethnicity']]

y = DfStudPerf['avg_scr']

#Transformación de variables categóricas:
X = pd.get_dummies(
    X,
    drop_first=True
)

## Train/Test

Este train/test utiliza el 80 porciento de los datos como datos de entrenamiento y el 20 porciento restante para poder evaluar el modelo(*test_size=0.2*). 

El parámetro *random_state=42* garantiza que la división sea siempre la misma, , que se obtengan siempre los mismos conjuntos al ejecutar el código nuevamente para obtener resultados reproducibles.

In [53]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Modelo: Regresión Lineal

In [54]:
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()
modelo.fit(X_train, y_train)

print("Intercepto:", modelo.intercept_)
print("Coeficiente:", modelo.coef_)

Intercepto: 68.27488153510335
Coeficiente: [-7.87769843  3.50210176 -4.65696265  1.92839373 -0.85410293 -3.27299591
  9.20765475 -4.09185541 -0.13983101  0.9179347   3.7808576   5.96021207]


El intercepto obtenido (68,27) representa el valor base estimado del promedio de calificaciones (y), cuando las variables predictivas (X) tienen valor 0. 

Los coeficientes indican cuánto aumenta o disminuye la predicción cuando una determinada característica está presente. 
Por poner un ejemplo, podemos observar que el coeficiente asociado a no haber realizado el curso de preparación es negativo (-7,88); esto implica que, en estos casos, se espera una disminución del promedio de notas. Disponer de almuerzo estándar presenta un coeficiente positivo (9,2), por lo cual podemos asociarlo con un mejor promedio académico.

## Métricas obtenidas

In [55]:
y_pred = modelo.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R²:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

R²: 0.16217185763155217
MAE: 10.490182374209294
RMSE: 13.401579844788277


In [59]:
comparacion = pd.DataFrame({
    'Real': y_test,
    'Valor predicho': y_pred
})

print(comparacion.head(10))

          Real  Valor predicho
521  87.000000       70.522773
737  64.000000       67.280948
740  75.000000       72.795942
660  74.666667       56.369159
411  81.666667       78.496790
678  78.000000       60.086185
626  67.333333       64.043195
513  59.000000       74.069709
859  77.333333       57.223262
136  48.666667       69.977854


Se generó un DataFrame llamado comparacion para visualizar la efectividad del modelo. La columna *Real* contiene los promedios de notas observados en el conjunto de prueba (y_test), mientras que la columna *Valor predicho* contiene los valores estimados por el modelo (y_pred).

Se observa que en algunos casos las predicciones tienen similitud con los valores de prueba (72,79/75), mientras que en otros presentan diferencias notorias (48,66/69,97). 

Esto coincide con las métricas obtenidas anteriormente (R² = 0,162; MAE = 10,49).

El modelo es útil para identificar patrones generales en los datos pero podríamos decir que su precisión predictiva es limitada. Probablemente existan factores relevantes para el promedio de notas que no están representados en las variables utilizadas.

## Interpretación

Las métricas obtenidas muestran que el modelo posee una capacidad predictiva moderada. 

El valor de R² = 0,162 indica que las variables seleccionadas explican aproximadamente el 16,2% de la variabilidad en el promedio de las calificaciones. 

El MAE = 10,49 y el RMSE = 13,40 reflejan errores de predicción elevados. Existe una gran proporción del rendimiento académico que no puede ser explicada por el modelo. 

Me parece que hay variables clave para el rendimiento académido, que no están contempladas dentro de este dataset. Sería interesante poder analizar factores como hábitos de estudio, asistencia, entorno familiar o nivel socioeconómico.

Este modelo que hizo basándose en la hipótesis 2, que buscaba determinar la relación de completar el curso de ingreso con las notas obtenidas. La evidencia apoya la hipótesis alternativa de que los estudiantes que completaron el curso obtienen mejores calificaciones que quienes no lo realizaron. 

El modelo asignó un efecto negativo a la ausencia del curso de preparación, aunque también evidenció que el rendimiento depende de múltiples factores adicionales.

El dataset utilizado puede contener sesgos asociados a variables demográficas y sociales, como género, nivel educativo de los padres o grupo étnico (grupos anonimizados). Su agregaron a X porque su inclusión mejora la capacidad predictiva del modelo. Estas predicciones no deberían utilizarse para etiquetar o discriminar estudiantes, sino únicamente con fines analíticos y educativos.
